# Course 9 · Machine Learning with Python
**Supervised:** regression (predict a number) and classification (predict a class).
**Unsupervised:** clustering and dimensionality reduction.
Target here: predict whether an application is **rejected**. Every line has a `#` comment.

## Step 1 · Prepare data

In [ ]:
# os lets Python talk to the operating system (files, folders).
import os
# If the practice data file is not in this folder yet...
if not os.path.exists("applications.csv"):
    # ...run make_data.py to create applications.csv and centers.csv.
    %run make_data.py
# pandas = tables (nickname pd); numpy = fast math (nickname np). These nicknames are standard.
import pandas as pd, numpy as np

# Load, de-duplicate, tidy labels, fill missing ages, add center info (same as earlier notebooks).
df = (pd.read_csv("applications.csv").drop_duplicates()
        .assign(channel=lambda d: d.channel.str.strip().str.lower(),
                age=lambda d: d.age.fillna(d.age.median()))
        .merge(pd.read_csv("centers.csv"), on="center_id"))
# Share of each class. If one class is rare (here about 20% rejected), the data is IMBALANCED.
print(df["rejected"].value_counts(normalize=True).round(3))

## Step 2 · Train/test split and a preprocessing pipeline
The scaler learns only from training data. This prevents *data leakage* (test information sneaking into training).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer      # apply different steps to different columns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

num_cols = ["age", "staff", "biometric_quality", "docs_complete"]   # numeric inputs
cat_cols = ["channel", "region"]                                    # text (categorical) inputs
X = df[num_cols + cat_cols]                       # all inputs
y = df["rejected"]                                # target: 1 = rejected, 0 = accepted

# stratify=y keeps the same rejected/accepted ratio in train and test sets.
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=4)

prep = ColumnTransformer([
    ("num", StandardScaler(), num_cols),                         # scale numbers to mean 0, std 1
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),   # text -> 0/1 columns; ignore unseen values
])
print(X_tr.shape, X_te.shape)                     # (rows, columns) of each set

## Step 3 · K-Nearest Neighbours: finding the best k
KNN labels a new case by majority vote of its k most similar training cases.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

ks = list(range(1, 26, 2))                        # odd k values 1,3,...,25 (odd avoids tied votes)
scores = []                                       # will hold the F1 score for each k
for k in ks:
    # A pipeline: preprocess, then classify. .fit() trains both steps.
    p = Pipeline([("prep", prep), ("clf", KNeighborsClassifier(n_neighbors=k))]).fit(X_tr, y_tr)
    scores.append(f1_score(y_te, p.predict(X_te)))   # F1 balances precision and recall
plt.plot(ks, scores, marker="o")
plt.xlabel("k"); plt.ylabel("F1"); plt.title("Choosing k")
plt.show()
best_k = ks[scores.index(max(scores))]            # the k with the highest F1
print("best k =", best_k)

## Step 4 · Decision tree, and how to read it
Each box asks a yes/no question about one feature; follow left (True) or right (False) down to a prediction.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# max_depth=3 keeps the tree small enough to read; criterion="entropy" = how splits are chosen.
tree = Pipeline([("prep", prep),
                 ("clf", DecisionTreeClassifier(max_depth=3, criterion="entropy", random_state=0))]
               ).fit(X_tr, y_tr)

plt.figure(figsize=(18, 7))
plot_tree(tree.named_steps["clf"],                                       # the fitted tree inside the pipeline
          feature_names=tree.named_steps["prep"].get_feature_names_out(), # names after preprocessing
          class_names=["accepted", "rejected"],
          filled=True, fontsize=8)                                       # colour boxes by majority class
plt.show()

## Step 5 · Logistic regression: probabilities and coefficients

In [ ]:
from sklearn.linear_model import LogisticRegression

# C = inverse of regularisation strength (smaller C = simpler model). max_iter = training steps allowed.
logreg = Pipeline([("prep", prep), ("clf", LogisticRegression(C=1.0, max_iter=1000))]).fit(X_tr, y_tr)

# predict_proba returns [P(accepted), P(rejected)] per row; [:5, 1] = first 5 rows, column 1.
print("P(rejected) for first 5 test rows:", logreg.predict_proba(X_te)[:5, 1].round(2))

# Coefficients: positive pushes toward "rejected", negative toward "accepted".
names = logreg.named_steps["prep"].get_feature_names_out()
coef = pd.Series(logreg.named_steps["clf"].coef_[0], index=names)
coef.sort_values().plot(kind="barh", figsize=(7, 5), title="Logistic regression coefficients")
plt.show()

## Step 6 · Compare five classifiers
With imbalanced classes, **accuracy misleads** (always guessing "accepted" is about 80% accurate). Compare F1, Jaccard, log loss and ROC-AUC too.

In [ ]:
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import jaccard_score, log_loss, roc_auc_score

# A dictionary of name -> model. Each model gets the same preprocessing.
models = {
    "KNN": KNeighborsClassifier(n_neighbors=best_k),
    "DecisionTree": DecisionTreeClassifier(max_depth=4, criterion="entropy", random_state=0),
    "LogReg": LogisticRegression(max_iter=1000),
    "SVM": CalibratedClassifierCV(SVC(kernel="rbf")),    # calibration makes SVM output probabilities
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=6, random_state=0),  # 200 trees vote
}
rows, fitted = [], {}                                     # results table rows; trained pipelines
for name, model in models.items():                        # .items() gives (name, model) pairs
    pipe = Pipeline([("prep", prep), ("clf", model)]).fit(X_tr, y_tr)
    pred = pipe.predict(X_te)                             # 0/1 predictions
    proba = pipe.predict_proba(X_te)[:, 1]                # probability of "rejected"
    fitted[name] = pipe                                   # keep it for later
    rows.append({"model": name,
                 "accuracy": accuracy_score(y_te, pred),  # share of all predictions that are right
                 "f1": f1_score(y_te, pred),              # balance of precision and recall (0..1)
                 "jaccard": jaccard_score(y_te, pred),    # overlap of predicted and true "rejected"
                 "log_loss": log_loss(y_te, proba),       # penalises confident wrong probabilities (lower = better)
                 "roc_auc": roc_auc_score(y_te, proba)})  # ranking quality (0.5 = random, 1 = perfect)
results = pd.DataFrame(rows).set_index("model").round(3).sort_values("f1", ascending=False)
results

## Step 7 · Confusion matrix and classification report for the best model

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

best = results.index[0]                           # top row of the sorted results = best F1
# Confusion matrix: rows = true class, columns = predicted class. The diagonal = correct.
ConfusionMatrixDisplay.from_estimator(fitted[best], X_te, y_te, display_labels=["accepted", "rejected"])
plt.title(best)
plt.show()
# Precision = of those flagged rejected, how many really were. Recall = of real rejections, how many we caught.
print(classification_report(y_te, fitted[best].predict(X_te), target_names=["accepted", "rejected"]))

## Step 8 · Tune hyperparameters with GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

rf = Pipeline([("prep", prep), ("clf", RandomForestClassifier(random_state=0))])
# Every combination below is tried: 2 x 3 x 2 = 12 settings, each with 5-fold cross-validation.
param_grid = {"clf__n_estimators": [100, 300],          # number of trees
              "clf__max_depth": [4, 8, None],           # how deep each tree may grow (None = no limit)
              "clf__class_weight": [None, "balanced"]}  # "balanced" gives the rare class more weight
grid = GridSearchCV(rf, param_grid, cv=5, scoring="f1", n_jobs=-1)   # n_jobs=-1 = use all CPU cores
grid.fit(X_tr, y_tr)
print(grid.best_params_,
      "| CV F1:", round(grid.best_score_, 3),
      "| test F1:", round(f1_score(y_te, grid.predict(X_te)), 3))

## Step 9 · Regression model: predict processing days

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

yr = df["processing_days"]                        # a numeric target this time
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X, yr, test_size=0.2, random_state=4)
reg = Pipeline([("prep", prep),
                ("rf", RandomForestRegressor(n_estimators=300, random_state=0))]).fit(Xr_tr, yr_tr)
p = reg.predict(Xr_te)
# MAE = average error in days; R² = share of variation explained.
print("MAE:", round(mean_absolute_error(yr_te, p), 2), "days | R²:", round(r2_score(yr_te, p), 3))

## Step 10 · K-Means clustering of centers (unsupervised)
No target: the algorithm groups similar centers by itself.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# One row per center: volume, average days, rejection rate, staff.
cf = df.groupby("center_id").agg(apps=("app_id", "count"), days=("processing_days", "mean"),
                                 reject=("rejected", "mean"), staff=("staff", "first"))
Z = StandardScaler().fit_transform(cf)            # scale so no feature dominates the distances

inertia, sil = [], []
for k in range(2, 8):                             # try 2..7 clusters
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z)   # n_init = restarts, keep the best
    inertia.append(km.inertia_)                   # within-cluster spread (lower = tighter)
    sil.append(silhouette_score(Z, km.labels_))   # separation between clusters (-1..1, higher = better)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(range(2, 8), inertia, marker="o"); ax[0].set_title("Elbow: look for the bend")
ax[1].plot(range(2, 8), sil, marker="o");     ax[1].set_title("Silhouette (higher = better)")
plt.show()

In [ ]:
from sklearn.decomposition import PCA

# Assign each center to one of 3 clusters, then describe each cluster by its averages.
cf["segment"] = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(Z)
display(cf.groupby("segment").mean().round(2))

# PCA squeezes 4 features into 2 "principal components" so we can draw the clusters on a flat chart.
xy = PCA(n_components=2).fit_transform(Z)
plt.scatter(xy[:, 0], xy[:, 1], c=cf["segment"], cmap="viridis", s=60)   # colour = cluster
plt.title("Center segments (PCA view)"); plt.xlabel("PC1"); plt.ylabel("PC2")
plt.show()

## ✍️ Your turn
1. Precision or recall: which matters more at an intake desk flagging risky applications? Re-run the grid with `scoring="recall"`.
2. Try `DBSCAN` (from `sklearn.cluster`) instead of K-Means on `Z`. How many clusters does it find?

In [ ]:
# your answer here